# Notebook 5 — Single-Head Masked Self-Attention

이제 GPT의 핵심 아이디어인 **masked self-attention**을 추가합니다.

이번 단계에서는 **single-head**에만 집중합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))

## 1. Single-head masked self-attention

In [ ]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim, block_size):
        super().__init__()
        self.key = nn.Linear(emb_dim, emb_dim, bias=False)
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)

        out = wei @ v
        return out

## 2. Attention 포함 최소 모델

In [ ]:
class TinyAttentionLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.attn = SingleHeadSelfAttention(emb_dim, block_size)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)
        pos = self.position_embedding(pos)[None]
        h = tok + pos
        h = self.attn(h)
        logits = self.lm_head(h)
        return logits

model = TinyAttentionLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 32, 65])


## 3. 학습

In [ ]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyAttentionLM(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 2.9403
epoch  1 | train loss 2.5259
epoch  2 | train loss 2.4235
epoch  3 | train loss 2.3847
epoch  4 | train loss 2.3608
epoch  5 | train loss 2.3433
epoch  6 | train loss 2.3321
epoch  7 | train loss 2.3190
epoch  8 | train loss 2.3157
epoch  9 | train loss 2.3073
epoch 10 | train loss 2.3027
epoch 11 | train loss 2.2995
epoch 12 | train loss 2.2979
epoch 13 | train loss 2.2887
epoch 14 | train loss 2.2884
epoch 15 | train loss 2.2866
epoch 16 | train loss 2.2843
epoch 17 | train loss 2.2800
epoch 18 | train loss 2.2780
epoch 19 | train loss 2.2756
epoch 20 | train loss 2.2747
epoch 21 | train loss 2.2757
epoch 22 | train loss 2.2732
epoch 23 | train loss 2.2741
epoch 24 | train loss 2.2692
epoch 25 | train loss 2.2675
epoch 26 | train loss 2.2674
epoch 27 | train loss 2.2706
epoch 28 | train loss 2.2655
epoch 29 | train loss 2.2660
epoch 30 | train loss 2.2629
epoch 31 | train loss 2.2634
epoch 32 | train loss 2.2638
epoch 33 | train loss 2.2606
epoch 34 | tra

## 4. Sampling

In [ ]:
@torch.no_grad()
def sample_attention_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_attention_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400))

ROMEO:
He my det ty ma bly boerden?

AUn rde sut theol,
The mey peant corres mavoe oud Riged atintatech: Ciz:
Beleanlld than an are. ovevefo Wentr:
Unne, alm
Eweasownetu Halto hey!
WAst wor nogit cke red me houp Apro forf to won, amno waurdd I wothircadsoeteve manto!

VINGOFR IOLOLEO:
wingu lp ed she youred.

Patr thoru the bran nd ayu igeve I VNanik I Whe bude, ry ancat meagoof thadeck whee sache ip; a


## 5. 정리

- 각 위치는 이전 위치들을 참조할 수 있습니다.
- 미래는 causal mask로 차단됩니다.
- 이제 모델이 어떤 위치를 참고할지 스스로 결정하기 시작합니다.